# 02. 설정 → TodaysMarket: 구축을 한 줄씩

| 단계 | QuantTool 코드 | ORE 원본 (`references/Engine/OREData/ored/`) |
|---|---|---|
| 레시피 읽기 | `data/configuration/`, `configs/` | `configuration/`, `todaysmarket.xml`, `curveconfig.xml`, `conventions.xml` |
| 시세 읽기 | `data/marketdata/loader.py` `CSVLoader`, `marketdatum.py` | `marketdata/loader.hpp`, `csvloader.hpp`, `marketdatumparser.cpp` |
| 빌더 찾기 | `data/utilities/registry.py` `BUILDERS` | (C++ 클래스 계층) |
| 순서 정하기 | `data/marketdata/dependencygraph.py` | `marketdata/dependencygraph.hpp` |
| 보관소 | `data/marketdata/market.py` `Market`, `MarketImpl` | `marketdata/market.hpp`, `marketimpl.hpp` |
| 구축 | `data/marketdata/todaysmarket.py` `TodaysMarket` | `marketdata/todaysmarket.cpp` (`initialise`, `buildNode`) |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 ORE `market.txt` 형식(`날짜 키 값`)의 **합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "tests" / "fixtures" / "market_ust_sample.txt"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.data.configuration import (
    InstrumentConventions,
    MarketObject,
    load_config,
    parse_curve_spec,
)
from quanttool.data.marketdata import CSVLoader, TodaysMarket
from quanttool.data.marketdata.dependencygraph import DependencyGraph
from quanttool.data.utilities.registry import BUILDERS

ASOF = date(2026, 10, 23)

## 1. 레시피 (무엇을 어떤 이름으로, 어떻게 만들지)
- `configs/market/todaysmarket.yaml` → `TodaysMarketParameters`: configuration → {시장에서 부르는 이름: 커브 지정 `Yield/USD/UST`}
- `configs/market/curves/ust.yaml` → `YieldCurveConfig`: 세그먼트마다 시세 키 목록 + 관례 id, 보간
- `configs/conventions/rates.yaml` → 관례. `load_config`가 전역 `InstrumentConventions`에 등록한다

In [ ]:
cfg = load_config(ROOT / "configs")
tmp = cfg.todaysmarket
print("configurations:", list(tmp.configurations))
print("DiscountCurve 매핑:", tmp.mapping(MarketObject.DiscountCurve))
print("YieldCurve 매핑   :", tmp.mapping(MarketObject.YieldCurve))
spec = parse_curve_spec(tmp.curve_specs()[0])
print("만들 레시피:", tmp.curve_specs(), "→", spec)

In [ ]:
recipe = cfg.curve_configs[spec.curve_config_id]
print(recipe.curve_id, recipe.currency, recipe.interpolation_variable, recipe.interpolation_method)
for seg in recipe.segments:
    print(seg.type, seg.conventions, seg.quotes[0], "…", seg.quotes[-1])
InstrumentConventions.instance().conventions().get("UST").model_dump()

## 2. 시세 읽기 → MarketDatum
Loader는 그 날짜의 시세를 `MarketDatum` 목록으로 준다(ORE `market.txt`의 `날짜 키 값`). MarketDatum은 값을 `SimpleQuote`로 들고 있고, 커브 헬퍼가 이 Quote에 연결된다. 와일드카드로 묶어 꺼낼 수도 있다.

In [ ]:
loader = CSVLoader(SAMPLE)
print("날짜:", loader.asof_dates())
display(
    pd.DataFrame(
        [
            (q.name, q.instrument_type, q.quote_type, q.tenor, q.value)
            for q in loader.load_quotes(ASOF)
        ],
        columns=["name", "instrument", "quote_type", "tenor", "value"],
    ).head()
)
[q.tenor for q in loader.get_wildcard("BOND/YIELD/USD/UST/PAR/*Y", ASOF)]

## 3. 빌더 찾기와 구축 순서
레시피 종류(`YieldCurve`)로 등록부에서 빌더를 찾는다. 레시피가 다른 레시피를 할인에 쓰면(`discount_curve`) 의존 그래프가 순서를 정한다(ORE: EUR6M은 EUR1D 다음). 지금은 UST 하나뿐이다.

In [ ]:
print("등록된 빌더:", BUILDERS.keys())
builder = BUILDERS["YieldCurve"]()
graph = DependencyGraph({("YieldCurve", "UST"): builder.requires(recipe)})
graph.order([("YieldCurve", "UST")])

## 4. TodaysMarket 생성자가 하는 일 (ORE `TodaysMarket::initialise`)
1. configuration마다 필요한 커브 지정(`Yield/USD/UST`) 목록을 만든다
2. 평가일을 asof로 두고 의존 순서대로 레시피를 구축한다. 같은 레시피는 한 번만(ORE `requiredYieldCurves_`)
3. 구축 직후 부트스트랩하고 **얼린다**(`freeze`)
4. 매핑의 이름마다 그 커브 Handle을 보관소 `(configuration, 용도, 이름)`에 둔다

In [ ]:
market = TodaysMarket(ASOF, cfg.todaysmarket, loader, cfg.curve_configs)
# 같은 것: TodaysMarket.from_config(ASOF, cfg, loader)
print("Yield 이름", market.yield_curve_names(), " 헬퍼 수", len(market.curve_helpers("UST")))
print("기준일", market.yield_curve("UST").referenceDate())
print(
    'discount_curve("USD")와 yield_curve("UST")는 같은 Handle:',
    market.discount_curve("USD") is market.yield_curve("UST"),
)

## 5. Market 규격으로 꺼내 쓰기
가격결정 코드는 시장에서 **이름으로** 커브를 꺼낸다(원칙 3). configuration을 주면 그 묶음에서 찾고, 없으면 `default`에서 찾는다(ORE와 같음).

In [ ]:
curve = market.yield_curve("UST")
d = ql.Date(17, 3, 2031)
print("DF", curve.discount(d))
print("zero", curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate())
print("pricing configuration → default로 대체:", market.yield_curve("UST", "pricing").discount(d))

## 6. 진단과 커브 표
`data/marketdata/diagnostics.py`, `analytics/app/marketmonitor.py`

In [ ]:
from quanttool.analytics.app import curve_shape, curve_table
from quanttool.data.marketdata.diagnostics import helper_repricing

helper_repricing(market, "UST")

In [ ]:
table = curve_table(market, "UST", cfg)
display((table[["par", "zero_cc", "fwd_1y"]] * 100).round(3))
curve_shape(table)

## 7. 얼린 시장은 다른 날짜 시장과 독립이다
날짜가 다른 시장을 여러 개 만들어도 각 시장의 커브는 자기 asof에 고정되어 있다. 다만 **상품 평가**는 전역 평가일을 쓰므로 각 시장의 `active()` 안에서 한다(03번).

In [ ]:
prev = TodaysMarket.from_config(date(2026, 10, 22), cfg, loader)
for m in (prev, market):
    c = m.yield_curve("UST")
    print(
        m.asof,
        "기준일",
        c.referenceDate(),
        " 10Y zero",
        round(c.zeroRate(10.0, ql.Continuous).rate() * 100, 4),
    )

다음: `03_trade_to_npv.ipynb` — 계약서를 이 시장에 붙여 가격을 낸다.